In [0]:
from pyspark.sql.functions import col, count

orders_silver = spark.table(
    "workspace.default.orders_silver"
)

In [0]:
def run_quality_check(check_name, check_condition, error_message):
    
    if not check_condition:
        print(f"❌ FAILED: {check_name}")
        raise Exception(error_message)
    
    print(f"✅ PASSED: {check_name}")

In [0]:
duplicate_orders = (
    orders_silver
    .groupBy("order_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
duplicate_count = duplicate_orders.count()

print("Duplicate orders:", duplicate_count)

In [0]:
if duplicate_count > 0:
    raise Exception(
        f"Data quality check failed: "
        f"{duplicate_count} duplicate order IDs found."
    )

print("Data quality check passed.")

In [0]:
required_columns = [
    "order_id",
    "customer_id",
    "order_status",
    "purchase_timestamp"
]

null_checks = {}

for column_name in required_columns:
    null_count = (
        orders_silver
        .filter(col(column_name).isNull())
        .count()
    )

    null_checks[column_name] = null_count

    print(f"{column_name}: {null_count}")

In [0]:
if any(count > 0 for count in null_checks.values()):
    raise Exception(
        f"Data quality check failed: NULL values found: {null_checks}"
    )

print("Required field check passed.")

In [0]:
customers_silver = spark.table(
    "workspace.default.customers_silver"
)

orders_without_customer = (
    orders_silver
    .join(
        customers_silver.select("customer_id"),
        on="customer_id",
        how="left_anti"
    )
)

missing_customer_count = orders_without_customer.count()

print("Orders without customer:", missing_customer_count)

In [0]:
if missing_customer_count > 0:
    raise Exception(
        f"Data quality check failed: "
        f"{missing_customer_count} orders have no matching customer."
    )

print("Referential integrity check passed.")

In [0]:
order_items_silver = spark.table(
    "workspace.default.order_items_silver"
)

products_silver = spark.table(
    "workspace.default.products_silver"
)

items_without_product = (
    order_items_silver
    .join(
        products_silver.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

missing_product_count = items_without_product.count()

print("Order items without product:", missing_product_count)

In [0]:
if missing_product_count > 0:
    raise Exception(
        f"Data quality check failed: "
        f"{missing_product_count} order items have no matching product."
    )

print("Product referential integrity check passed.")

In [0]:
def run_quality_check(check_name, check_condition, error_message):
    
    if not check_condition:
        print(f"❌ FAILED: {check_name}")
        raise Exception(error_message)
    
    print(f"✅ PASSED: {check_name}")

In [0]:
run_quality_check(
    "Duplicate order ID check",
    duplicate_count == 0,
    f"Found {duplicate_count} duplicate order IDs."
)

In [0]:
run_quality_check(
    "Required fields check",
    all(count == 0 for count in null_checks.values()),
    f"NULL values found: {null_checks}"
)

In [0]:
run_quality_check(
    "Order-customer referential integrity",
    missing_customer_count == 0,
    f"Found {missing_customer_count} orders without a customer."
)

In [0]:
items_without_product = (
    order_items_silver
    .join(
        products_silver.select("product_id"),
        on="product_id",
        how="left_anti"
    )
)

missing_product_count = items_without_product.count()

run_quality_check(
    "Order-item product referential integrity",
    missing_product_count == 0,
    f"Found {missing_product_count} order items without a matching product."
)

In [0]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger("ecommerce_pipeline")

In [0]:
def run_quality_check(check_name, check_condition, error_message):

    if not check_condition:
        logger.error(f"FAILED: {check_name}")
        raise Exception(error_message)

    logger.info(f"PASSED: {check_name}")

In [0]:
run_quality_check(
    "Duplicate order ID check",
    duplicate_count == 0,
    f"Found {duplicate_count} duplicate order IDs."
)

In [0]:
orders_bronze = spark.table(
    "workspace.default.orders_bronze"
)

In [0]:
expected_types = {
    "order_id": "string",
    "customer_id": "string",
    "order_status": "string"
}

In [0]:
for column_name, expected_type in expected_types.items():

    actual_type = dict(
        orders_bronze.dtypes
    )[column_name]

    print(
        f"{column_name}: "
        f"expected={expected_type}, "
        f"actual={actual_type}"
    )

In [0]:
for column_name, expected_type in expected_types.items():

    actual_type = dict(
        orders_bronze.dtypes
    )[column_name]

    run_quality_check(
        f"{column_name} schema check",
        actual_type == expected_type,
        f"{column_name} has type {actual_type}, "
        f"expected {expected_type}"
    )